# Protocol 1 — Cardiac Phase Windowing

This notebook reproduces the trial-windowing analysis for Protocol 1
(*Cardiac Phase Gating Paradigm*).  It demonstrates how APGI predicts
a cardiac-phase × ignition interaction mediated by interoceptive precision
$\Pi^i_{\text{eff}}$.

> **Note:** This notebook uses synthetic data that mirrors the protocol
> structure.  Replace `generate_synthetic_session` with your own data
> loader once real data are available.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from apgi.core import compute_pi_i_eff, compute_S_t, compute_theta_t, ignition_criterion
from apgi.normalizer import APGINormalizer
from apgi.clinical import EnhancedClinicalInterpreter

## 1 — Synthetic Protocol 1 session

We simulate 480 trials (8 blocks × 60) with two cardiac-phase conditions:

* **Systole** — interoceptive precision is lower: $\Pi^i = 0.6$
* **Diastole** — interoceptive precision is higher: $\Pi^i = 1.2$

Per the APGI protocol, these values come from `apgi_parameters` in
`protocols/protocol_1_cardiac_phase.json`.

In [ ]:
# Protocol 1 APGI parameters
KAPPA        = 100.0
ALPHA        = 0.3
BETA         = 0.7
PI_I_SYSTOLE  = 0.6
PI_I_DIASTOLE = 1.2

N_TRIALS = 480
rng = np.random.default_rng(2025)

# Alternate systole / diastole trials
cardiac_phase = np.array(['systole', 'diastole'] * (N_TRIALS // 2))
rng.shuffle(cardiac_phase)

pi_e          = rng.uniform(0.8, 1.5, N_TRIALS)
z_e           = rng.uniform(0.2, 1.0, N_TRIALS)
z_i           = rng.uniform(0.1, 0.8, N_TRIALS)
C_metabolic   = rng.uniform(0.5, 2.0, N_TRIALS)
V_information = rng.uniform(0.1, 1.0, N_TRIALS)

pi_i = np.where(cardiac_phase == 'systole', PI_I_SYSTOLE, PI_I_DIASTOLE)

# Compute APGI quantities trial-by-trial
pi_i_eff  = compute_pi_i_eff(pi_i, C_metabolic, kappa=KAPPA)
S_t       = np.array([compute_S_t(pi_e[i], z_e[i], pi_i_eff[i], z_i[i]) for i in range(N_TRIALS)])
theta_t   = np.array([compute_theta_t(C_metabolic[i], V_information[i], ALPHA, BETA) for i in range(N_TRIALS)])
ignition  = np.array([ignition_criterion(S_t[i], theta_t[i]) for i in range(N_TRIALS)])

print(f"Overall ignition rate: {ignition.mean():.3f}")

## 2 — Cardiac-phase comparison

In [ ]:
mask_sys = cardiac_phase == 'systole'
mask_dia = cardiac_phase == 'diastole'

rate_sys = ignition[mask_sys].mean()
rate_dia = ignition[mask_dia].mean()
mean_S_sys = S_t[mask_sys].mean()
mean_S_dia = S_t[mask_dia].mean()

print(f"Systole  — ignition rate: {rate_sys:.3f}   mean S_t: {mean_S_sys:.3f}")
print(f"Diastole — ignition rate: {rate_dia:.3f}   mean S_t: {mean_S_dia:.3f}")
print(f"Delta (diastole − systole): {rate_dia - rate_sys:+.3f}")

## 3 — Visualise $S_t$ distributions by cardiac phase

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (mask, label, colour) in zip(
    axes,
    [(mask_sys, 'Systole', '#d6604d'), (mask_dia, 'Diastole', '#2166ac')]
):
    ax.hist(S_t[mask], bins=30, color=colour, alpha=0.8, edgecolor='white')
    ax.axvline(theta_t[mask].mean(), color='black', lw=1.5, ls='--', label=r'mean $\theta_t$')
    ax.set_title(f"{label}  (ignition rate = {ignition[mask].mean():.2f})", fontsize=11)
    ax.set_xlabel(r'$S_t$'); ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Protocol 1 — Cardiac Phase × Ignition', fontsize=12)
plt.tight_layout()
plt.show()

## 4 — Clinical interpretation with `EnhancedClinicalInterpreter`

Slide a 20-trial window over the session and label each window.

In [ ]:
interp = EnhancedClinicalInterpreter(window_size=20)
reports = interp.interpret_session(S_t, theta_t, C_metabolic)

levels = [r.level.value for r in reports]
unique, counts = np.unique(levels, return_counts=True)
for lv, cnt in zip(unique, counts):
    print(f"  {lv:<25} {cnt} windows  ({cnt/len(reports)*100:.0f}%)")